In [4]:
import pandas as pd
import sys
import os
from pathlib import Path

current_dir = Path(os.getcwd())
if 'notebooks' in str(current_dir):
    project_root = current_dir.parent.parent
else:
    project_root = current_dir
    while project_root != project_root.parent:
        if (project_root / 'src').exists():
            break
        project_root = project_root.parent

project_root_str = str(project_root.resolve())

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.fetch_player_stats import *
from src.utils.fetch_team_stats import *
from src.utils.helper_functions import *

## Fetches Player Gamelogs

In [ ]:
nba = FetchPlayersStats()
data = nba.getCompleteStats(
    season='2025-26', 
    season_type='Regular Season', 
    sleep_time=2, 
    max_workers=5,
    batch_limit=100,
    complete_cache_file='data/raw/seaon_stats/S26.csv',
    include_playbyplay=False
)
data.tail()

Processing 9 games (limited by batch_limit=100)

Processing batch 1/1 (9 games)
Completed batch 1/1

Merging team stats...
Cache updated. Total games now: 323


,PLAYER_NAME,PLAYER_ID,MATCHUP,TEAM_ABBREVIATION,TEAM_ID,OPP_ABBREVIATION,HOME_GAME,GAME_ID,GAME_DATE,WL,...,OPP_OFF_RATING,OPP_PTS,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TOV
7085,Brandon Williams,1630314,DAL vs. MIA,DAL,1610612742,MIA,1,0022500332,2025-12-03,W,...,102.0,108.0,40.0,101.0,0.396,41.0,29.0,11.0,5.0,11.0
7086,Nikola Jović,1631107,MIA @ DAL,MIA,1610612748,DAL,0,0022500332,2025-12-03,L,...,116.5,118.0,46.0,91.0,0.505,50.0,31.0,6.0,7.0,15.0
7087,D'Angelo Russell,1626156,DAL vs. MIA,DAL,1610612742,MIA,1,0022500332,2025-12-03,W,...,102.0,108.0,40.0,101.0,0.396,41.0,29.0,11.0,5.0,11.0
7088,Daniel Gafford,1629655,DAL vs. MIA,DAL,1610612742,MIA,1,0022500332,2025-12-03,W,...,102.0,108.0,40.0,101.0,0.396,41.0,29.0,11.0,5.0,11.0
7089,Luke Kennard,1628379,ATL vs. LAC,ATL,1610612737,LAC,1,0022500327,2025-12-03,L,...,118.8,115.0,44.0,89.0,0.494,54.0,29.0,12.0,1.0,12.0


In [9]:
pd.set_option('display.max_columns', None)

s19_regular = pd.read_csv('data/raw/seaon_stats/S19.csv')
s20_regular = pd.read_csv('data/raw/seaon_stats/S20.csv')
s21_regular = pd.read_csv('data/raw/seaon_stats/S21.csv')
s22_regular = pd.read_csv('data/raw/seaon_stats/S22.csv')
s23_regular = pd.read_csv('data/raw/seaon_stats/S23.csv')
s24_regular = pd.read_csv('data/raw/seaon_stats/S24.csv')
s25_regular = pd.read_csv('data/raw/seaon_stats/S25.csv')
s26_regular = pd.read_csv('data/raw/seaon_stats/S26.csv')

for df in [s20_regular, s21_regular, s22_regular, s23_regular, s24_regular, s25_regular, s26_regular]:
    df.drop(columns=['Unnamed: 0', 'DEF_FG_PCT_ALLOWED', 'DEF_3PT_PCT_ALLOWED', 'PTS_ALLOWED_PER_MIN', 
                     'DEF_TOV_FORCED_PER_MIN', 'DEF_BLOCKS_PER_MIN', 'DEF_SHOOTING_FOULS_PER_MIN', 
                     'DEF_AST_ALLOWED_PER_MIN'], errors='ignore', inplace=True)

## Fetching play by play data for model v2

In [4]:
# nba = FetchPlayersStats()
# data = nba.getCompleteStats(
#     season='2023-24', 
#     season_type='Regular Season', 
#     sleep_time=2, 
#     max_workers=5,
#     batch_limit=50,
#     complete_cache_file='../DATA/CSV_FILES/REGULAR_DATA/S24v2.csv',
#     include_playbyplay=True
# )
# data.tail()

## Assign features for regular season data

In [ ]:
# data = [s22_regular,s23_regular,s24_regular, s25_regular, s26_regular]
# seasons = [2022,2023,2024, 2025, 2026]
data = [s26_regular]
seasons = [2026]

output_dir = os.path.join(project_root, 'data', 'processed', 'training')
os.makedirs(output_dir, exist_ok=True)

for season_data, year in zip(data, seasons):
    print(f"Processing year {year}...")
    
    # Process features
    processed_data = process_season_features(
        season_data, 
        prop_type='PTS',
        year=year
    )
    
    # Save file
    output_path = os.path.join(output_dir, f'PTS_TRAIN_{str(year)[-2:]}.csv')
    processed_data.to_csv(output_path)
    print(f"Completed {year}")

Processing year 2026...
Loading position cache...
No cache file found, starting fresh
Found 492 unique players, 492 need to be fetched
Fetching 492 new players using 4 threads...
Fetched PLAYER_ID 200768... (1/492)
Fetched PLAYER_ID 101108... (2/492)
Fetched PLAYER_ID 2544... (3/492)
Fetched PLAYER_ID 201142... (4/492)
Fetched PLAYER_ID 201143... (5/492)
Fetched PLAYER_ID 201144... (6/492)
Fetched PLAYER_ID 201145... (7/492)
Fetched PLAYER_ID 201566... (8/492)
